::: {.sb-lesson-links}
[Exercises →](../exercises/california-housing.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/02-linear-regression.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-computing-metrics.ipynb){download="lesson-computing-metrics.ipynb"}
:::

This notebook shows you two methods to compute the metrics used for performance evaluation and feature selection. 

### Fisher's F-statistic

A statistical test is a process that checks whether a hypothesis is supported or contradicted by the data at our disposal. This hypothesis, called the null hypothesis and denoted $H_{0}$, would have consequences on the properties of the observed data if it were true. These properties are summarized by a test statistic, whose value tells us how compatible the data are with $H_0$.

Fisher's F-statistic tests the following hypotheses:


* When the Fisher test is applied to the model as a whole, the null hypothesis $H_{0}$ is "the variables chosen to build the model are not jointly significant in describing the target variable". If this hypothesis is true, the F-statistic follows a Fisher distribution (F-distribution) with parameters $(p, n - p - 1)$, where $p$ is the number of explanatory variables and $n$ is the number of observations used to train the model. If the value of the F-statistic, denoted "F", falls outside the most probable regions of this distribution, we can reject the null hypothesis and conclude that the chosen model has real explanatory power on the target variable.

It may seem a little far-fetched, but all statistical tests work like that. We make an assumption; if this assumption held, the test statistic would follow a given distribution. If the actual value of the statistic lands too far from the probable values of this distribution, we are allowed to reject the null hypothesis.

Mathematically, the F-statistic is written:

$$
F = \frac{SSE / p}{SSR / (n - p - 1)}
$$

The F-test can also compare two nested models: model 1, which includes "model_1_variables", and model 2, which includes "model_1_variables + $X_d$" ($p_1$ and $p_2$ explanatory variables respectively). In this case, the F-statistic follows an F-distribution with parameters $(p_2 - p_1, n - p_2 - 1)$ if the hypothesis that the simpler model (model 1) is enough to describe the target variable is true. The formula of F is then:

$$
F = (\frac{SSR_{1}-SSR_{2}}{p_{2}-p_{1}})(\frac{n-p_{2}-1}{SSR_{2}})
$$

If the value of the F-statistic is in an unlikely region of the F-distribution, the hypothesis is rejected, and the test suggests that the more complex model 2 brings significant additional information compared to the simpler model 1.

Graphically the F-test can be illustrated as follows:

![F-statistic](../img/f_distribution.png)

The figure shows the density of the F-distribution. As in any test, we choose a level $\alpha$ between 0 and 1, which sets the size of the rejection zone. Very often, we choose $\alpha = 5\%$ when no specific knowledge helps us adjust our standards. The F-test is one-sided: only large values of F allow us to reject the hypothesis. More precisely, if the value of F falls in the upper 5% of the expected distribution, we can say that the hypothesis is rejected at the $1 - \alpha$ = 95% confidence level.

This first metric lets us test the hypothesis that the explanatory variables have no influence on the target variable. We will now look at metrics that indicate the performance level of the model.

* $R^2_{adjusted}$

$R^2_{adjusted}$ is a modified version of $R²$ that penalizes the number of explanatory variables selected to build the model. Its mathematical formula is:

$$
R^2_{adjusted} = 1-\frac{n-1}{n-p-1}(1-R^2)
$$

where $p$ is the number of explanatory variables and $n$ is the number of observations. The growth of $R^2$ as a function of $p$ is compensated by the growth of the penalty factor $\frac{n-1}{(n-p-1)}$, which multiplies $(1-R^2)$ and increases with $p$. Consequently, if an explanatory variable does not bring enough information, $R^2_{adjusted}$ decreases. This is why we can use this indicator to compare models that do not have the same number of explanatory variables.

* P-values

P-values evaluate the contribution of each explanatory variable individually, as opposed to evaluating the model as a whole. Unfortunately, they cannot easily be computed with sklearn, so we will introduce the statsmodels library, which calculates all the important metrics automatically. The p-value is the probability of observing a coefficient at least as far from 0 as the estimated one if the parameter's true value were 0, in other words if the variable did not bring any information to the linear model. A small p-value is therefore evidence that the variable is useful. Usually, we consider that a p-value below 5% means that the variable is significant, and that it is not significant otherwise. However, this depends on the context and the standards of your industry: for example, web marketing agencies typically have lower standards than pharmaceutical companies, because their goals and constraints are fundamentally different.


## Computation by hand

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
# SST, SSE and SSR have to be calculated manually
# generate some example data
X = np.array([
    [1,3,5,6,7],
    [4.6, 3.7, 3.4, 3.0, 3.1]
]).transpose()
Y = np.array([2.1, 3.5, 4.4, 5.6, 5.9])

from sklearn.linear_model import LinearRegression
model = LinearRegression() # create and instanceof the model

model.fit(X,Y) # fit the model

# calculate evaluation metrics
SST = np.sum(np.square(Y - np.mean(Y)))
print("Sum of Square Total {}".format(SST))

SSE = np.sum(np.square(model.predict(X) - np.mean(Y)))
print("Sum of Square Explained {}".format(SSE))

SSR = np.sum(np.square(Y - model.predict(X)))
print("Sum of Square Residual {}".format(SSR))
print("\n")

# calculate R square and adjusted R-square
R_2 = 1 - SSR/SST
print("R square {}".format(R_2))
R_2_alt = model.score(X,Y) # alternative method to calculate R square
print("R square {}".format(R_2_alt))
n = X.shape[0]
p = X.shape[1]
R_2_adj = 1 - (n-1)/(n-p-1)*(1-R_2)
print("R square adjusted {}".format(R_2_adj))


Sum of Square Total 9.74
Sum of Square Explained 9.612353658536577
Sum of Square Residual 0.1276463414634147


R square 0.9868946261331196
R square 0.9868946261331196
R square adjusted 0.9737892522662392


## Computation with statsmodels

In [ ]:
# alternative solution with library statsmodels (useful mainly for linear models)
import statsmodels.api as sm

X2 = sm.add_constant(X) # the coefficient beta_0 also called intercept is not automatically included, so we need to manually add a constant variable equal to one.
est = sm.OLS(Y, X2)
est2 = est.fit()
print("\n")
print("-----------------------------------------------------------------------------------------")
print("------------------------Results from statsmodels-----------------------------------------")
print("-----------------------------------------------------------------------------------------")
print("\n")
print(est2.summary())



-----------------------------------------------------------------------------------------
------------------------Results from statsmodels-----------------------------------------
-----------------------------------------------------------------------------------------


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.974
Method:                 Least Squares   F-statistic:                     75.30
Date:                Mon, 16 Nov 2026   Prob (F-statistic):             0.0131
Time:                        19:02:14   Log-Likelihood:                 2.0751
No. Observations:                   5   AIC:                             1.850
Df Residuals:                       2   BIC:                            0.6781
Df Model:                           2                                         
Covariance Type

/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:3158: ValueWarning: omni_normtest is not valid with less than 8 observations; 5 samples were given.
  omni, omnipv = cached("omni_normtest", lambda: omni_normtest(self.wresid))


::: {.sb-next}
**[Your turn: Linear regression exercises →](../exercises/california-housing.ipynb)**
:::